# FracAtlas runner (Kaggle)

Thin runner: **all logic lives in the repo** (`src/`, `scripts/`). This notebook only
clones the repo, installs it, points it at the data, and calls scripts.

**Before running:** Settings → Internet **On**; Accelerator **GPU T4 x2** (or P100);
*Add Input* → your FracAtlas dataset. Phase 0 does not need the GPU yet.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Akinchan-Jain-7/fracatlas-fracture-detection.git"
WORKDIR = "/kaggle/working/fracatlas-fracture-detection"

# Clone once; on later runs in the same session, just pull the latest commits.
if os.path.isdir(WORKDIR):
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)
os.chdir(WORKDIR)
!git log --oneline -1

In [ ]:
# Editable install: imports resolve to the cloned source, so a `git pull` is
# picked up without reinstalling. Kaggle's preinstalled CUDA torch is kept.
!pip install -q -e ".[dev]"

In [ ]:
# Point the code at the read-only Kaggle input folder. The loader searches
# below this for dataset.csv, so the exact dataset slug does not matter.
os.environ["FRACDET_DATA_DIR"] = "/kaggle/input"
!python scripts/env_check.py

In [ ]:
!python scripts/verify_data.py

In [ ]:
!pytest -q

## Phase 1: data
Rebuilds the split from the raw images and checks it is **identical** to the
committed `splits/split_v1.csv` (proves reproducibility on this machine), then runs EDA.

In [ ]:
!python scripts/make_split.py --verify

In [ ]:
!python scripts/eda.py --out /kaggle/working/outputs/eda

from IPython.display import Image, display

for name in [
    "region_fracture_rate",
    "shortcut_attributes",
    "fracture_rate_by_id",
    "duplicate_distances",
]:
    display(Image(f"/kaggle/working/outputs/eda/{name}.png"))

## Phase 2: baseline training
Needs the **GPU** accelerator. ~10 min on a T4. The test split is never loaded.

In [ ]:
!python scripts/train.py --config configs/baseline_resnet50.yaml

In [ ]:
!python scripts/show_run.py --latest baseline_resnet50

## Phase 3: threshold on validation, then the test set once

**Do not run the Phase 2 training cell above**: Phase 3 evaluates the *existing*
Phase 2 model. Run the setup cells (clone, install, data path, verify, tests), then these.

**Add Input → Your Work → this notebook, Version 1** (the Phase 2 run with `best.pt`).
`--find-run` searches `/kaggle/input` for it, so the mount path does not matter.

### Step 1: choose the threshold (validation only, CPU is enough)

In [ ]:
!python scripts/tune_threshold.py --find-run baseline_resnet50

from IPython.display import Image, display

display(Image("outputs/eval/baseline_resnet50/tune/val_threshold_sweep.png"))

### Stop: pre-register the threshold

Download everything in `outputs/eval/baseline_resnet50/tune/` (Output panel), put it in
`results/phase3_baseline_resnet50/` in your local repo, then
`git add results/phase3_baseline_resnet50 && git commit -m "phase3: pre-register threshold" && git push`.

Only after the push, continue. Step 2 refuses to run if the threshold is not committed.

### Step 2: score the test set (GPU, ~2 min)

In [ ]:
!git pull --ff-only
!python scripts/evaluate_test.py --find-run baseline_resnet50

for name in ["pr_roc_val_vs_test", "confusion_test", "confusion_test_no_hardware"]:
    display(Image(f"outputs/eval/baseline_resnet50/test/{name}.png"))

Download `outputs/eval/baseline_resnet50/test/` into `results/phase3_baseline_resnet50/`,
commit and push. Then save a Kaggle version (Quick Save) so the outputs stay available.

### Fallback: no Kaggle Dataset attached
Downloads ~340 MB from Figshare into `/kaggle/working/data` (lost when the session ends).
Uncomment to use:

In [ ]:
# os.environ["FRACDET_DATA_DIR"] = "/kaggle/working/data"
# !python scripts/download_data.py
# !python scripts/verify_data.py